In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
import os
import sys
file_path = os.getcwd()
project_root = os.path.dirname(file_path)
print(project_root)
# Add the project root to the sys.path
if project_root not in sys.path:
    sys.path.append(project_root)
from dotenv import load_dotenv
# from langchain.agents import AgentExecutor  # legacy (langchain 0.x; not used — agents come from crunch_llm.code.agents)
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
# from langchain.agents import create_tool_calling_agent  # legacy (langchain 0.x; not used — agents come from crunch_llm.code.agents)
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.prompts import PromptTemplate
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings
from langchain_core.tools import tool
from typing import Dict, List, Union, Any 
from pathlib import Path
from IPython.display import display, Image
from typing import Optional,Dict, List, Tuple  
# Encoding
import base64
import mimetypes
import io  
import PIL.Image
import json 
import subprocess
import shutil # 
import re
from datetime import datetime 
import os
from pathlib import Path
# Our Libraries
from crunch_llm.code.agents import *
from crunch_llm.code.teams_SciC import *
from crunch_llm.code.utils import *

/users/jdtoscan/.conda/envs/Agent_Hub/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/oscar/data/gk/jdtoscan/LLM/Crunch_LLM


# Initialization

In [2]:
my_database_path ="/users/jdtoscan/data/jdtoscan/LLM/Crunch_LLM/Documents/database_NM"
my_database_path_functions="/users/jdtoscan/data/jdtoscan/LLM/Crunch_LLM/Documents/database_Modules_google"
my_database_path_functions_ollama="/users/jdtoscan/data/jdtoscan/LLM/Crunch_LLM/Documents/database_Modules_ollama"
database_name = 'Layers'
codes_database = 'Numerical_Methods_opt'

In [3]:
## Strategy Team
strategist_agent = memoryAgent(model='gemini-3-pro-preview', 
                     verbose=False,
                     system_prompt=strategist_agent_numerical_prompt,
                     model_provider='google',     
                    )
critic_agent = memoryAgent(model='gpt-5.1', 
                     verbose=False,
                     system_prompt=strategist_critic_numerical_prompt,
                     model_provider='openai',     
                    )

## Collection Team
retriever_agent = Retriever_Agent(model='gpt-oss:120b', 
                     verbose=False,
                     system_prompt=Retriever_system_prompt,
                     db_path=my_database_path,
                     collection_name=codes_database,
                     k=3)
validator_agent = memoryAgent(model='grok-4-1-fast-reasoning', 
                     verbose=False,
                     system_prompt=Validator_system_prompt,
                     model_provider='grok',     
                    )

# Implementation Team
planner_agent = memoryAgent(model='gemini-2.5-pro', 
                     verbose=False,
                     system_prompt=planner_agent_system_prompt_numerical,
                     model_provider='google',     
                    )
planner_parser_agent =  StructuredAgent(
                    model='gpt-oss:120b', 
                    system_prompt=planner_parser_system_prompt,
                    verbose=False
                )
patcher_agent = StructuredAgent(
                    model='gpt-oss:120b', 
                    system_prompt=patcher_agent_system_prompt_PINN,
                    verbose=False
                )

## Debugging Team

inspector_agent = StructuredAgent(
                    model='grok-4-1-fast-reasoning', 
                    model_provider='grok',
                    system_prompt=inspector_system_prompt,
                    verbose=False
                )


naming_agent = StructuredAgent(model='gpt-oss:120b', 
                     verbose=False,
                     system_prompt=naming_agent_system_prompt_PINN,
                    )


debugger_agent = StructuredAgent(model='claude-sonnet-4-5', 
                     verbose=False,
                     system_prompt=debugger_system_prompt_numerical,
                     model_provider='claude',     
                    )
## Advisor Team
advisor_agent = aRAGAgent_multimodal(
                    model='gemini-3-pro-preview', 
                    verbose=True,
                    system_prompt=advisor_system_prompt_numerical,
                    model_provider='google',     
                    embeddings_provider='google', 
                    embeddings_model='models/text-embedding-004',
                    db_path=my_database_path_functions,
                    collection_name=database_name,
                    k=5
                )
advisor_parser = StructuredAgent(
                    model='gpt-oss:120b', 
                     verbose=False,
                     system_prompt=advisor_parser_system_prompt_ON,
                    )

register_agent_parser =StructuredAgent(
                    model='grok-4-1-fast-reasoning', 
                    model_provider='grok',
                    system_prompt=register_agent_system_prompt_numerical,
                    verbose=False
                )


## User Query

In [4]:
PINN_parameter=525
saved_data_info="Saves 'results.npz' containing keys: 'model_name' (string), 'all_errors' (list of ~10000+ floats), 'all_its' (list), 'all_loss' (list), 'all_gamma' (list), 'all_lamB' (list), 'all_max_RBA' (list), 'all_L_infty' (list), 'all_L_2' (list), 'all_Re_est' (list of floats), 'Re' (scalar float), 'x_sensor' (500,), 'y_sensor' (500,), 'u_sensor' (500,), 'v_hidden_at_sensor' (500,).'"
path_data="/users/jdtoscan/data/jdtoscan/LLM/Crunch_LLM/Results/PINN_codes//Inverse_NavierStokes_Find_Re_HiddenField_v/2025_11_27_0058/"
user_question = f"""
Let's solve the inverse 2D steady-state incompressible Navier-Stokes equations (lid-driven cavity):

$$ \\nabla \\cdot \\mathbf{{u}} = 0 $$
$$ (\\mathbf{{u}} \\cdot \\nabla) \\mathbf{{u}} = -\\nabla p + 1/Re \\nabla^2 \\mathbf{{u}} $$

Domain: $(x,y) \\in [0,1] \\times [0,1]$; Here Re={PINN_parameter}.

Boundary Conditions:
- $$u(x,1) = 1 - \\frac{{\\cosh(C_0(x-0.5))}}{{\\cosh(0.5 C_0)}}$$ where the constant is set to $C_0 = 50$.
- Bottom wall (y=0): $\\mathbf{{u}} = (0, 0)$ (no-slip)
- Left wall (x=0): $\\mathbf{{u}} = (0, 0)$ (no-slip)
- Right wall (x=1): $\\mathbf{{u}} = (0, 0)$ (no-slip)

**Cross Validation**
 --- Sensors:

The data was saved as: 
path: {path_data}

details: {saved_data_info}

This file contains a list of sensors. Once the numerical solution is obtained, please evaluate it on the sensor points and report the relative L2 error of u and relative L2 error of v.

 --- True Data:

path: /users/jdtoscan/data/jdtoscan/LLM/Crunch_LLM/Documents/codes/Data/ldc_Re500.npz

The dataset is unstructured and was saved as an NPZ format containing keys: 'x', 'y', 'u', 'v', 'p'
- 'x', 'y': Node coordinates (1D arrays)
- 'u', 'v': Velocity components at each node (1D arrays, same length as x and y)
- 'p': Pressure at each node (1D array, same length as x and y)

The true solution is here; please report the L2 error for u and v of the true solution against the new numerical solution.

**Output:**
The final plot must be saved as `summary_all.png`.
1. Magnitude of velocity field from the new numerical simulation.
2. Error maps comparing the numerical solution to the True Data (Re=500).
3. A scatter plot comparing the PINN predictions at sensors vs. the Numerical Solution at those same points.
"""


In [5]:
checkpoint_project_dir=''
strategist_report, error_report, history_log_string, history, initial_code, filename= load_checkpoint_data(checkpoint_project_dir)

--- Starting a new run (no resume directory specified). ---


In [6]:

max_iterations_all=50
test_image_summary=False
refinement_iterations=0
is_approved = False
base_results_dir='/users/jdtoscan/data/jdtoscan/LLM/Crunch_LLM/Results/NM_codes/'
specific_requests=''#'\nSPECIFIC REQUESTS: QR-DeeNet with KKAN and RBF trained for 200000 iterations\n'# Pleaceholder for extra instructions
new_code_details='Test'
clean_functions=''
my_functions=''
requires_name=True
debug_approval=True
critic_approval = False
# --- Initialize history and log string OUTSIDE the loop ---
project_dir = "" # Initialize project_dir
critic_feedback=""
while (is_approved == False and refinement_iterations < max_iterations_all):
        print(f"\n--- Code Creation Iteration: {refinement_iterations} ---")

        # --- Print the history from the LAST completed run ---
        print("\n--- CURRENT TRIAL HISTORY ---")
        print(history_log_string)
        print("---------------------------\n")
        if debug_approval:
                print("--- 1. Calling Strategist Team ---")
                strategist_report, critic_approval=strategy_team(
                                                        user_question,
                                                        strategist_agent,
                                                        critic_agent,
                                                        history_log_string,
                                                        error_report,
                                                        max_iterations=5)
        else:
                print("--- There was an implementation error so we will keep the same strategy for this iteration---")
        if critic_approval:
                if initial_code=='':
                        print("--- 2. Calling Collection Tream ---")
                        initial_code, filename, _, _=retrieval_team(
                                                                        strategist_report,
                                                                        retriever_agent,
                                                                        validator_agent,
                                                                        max_iterations= 5)
                        print(f"\n--- Retrieved template: {filename} (First 5 lines) ---")
                        lines = initial_code.split('\n')
                        preview = '\n'.join(lines[:5])
                        print(preview)
                        print("---------------------------------------------------\n")
                print("--- 3. Calling Implementation Team ---")
                filename,initial_code=implementation_team(
                                                                user_question,
                                                                planner_agent,
                                                                patcher_agent,
                                                                planner_parser_agent,
                                                                initial_code,
                                                                filename,
                                                                error_report,
                                                                strategist_report,
                                                                remove_comments=False
                                                                )
                print("--- 4. Calling Debugging Team ---")
                prompt_inspector = (
                f"[SCIENTIFIC_STRATEGY_REPORT]:\n{strategist_report}\n\n"
                f"[FINAL_GENERATED_CODE]:\n{initial_code}\n\n"
                )
                response_inspector=inspector_agent.run(prompt_inspector)
                implementation_details=response_inspector['summary']
                generated_data_details=response_inspector['data_saving']
                implementation_approval=check_approval(response_inspector['status'])
                print('Inspector summary:\n',implementation_details)
                if implementation_approval:

                        project_dir,filename,final_filepath,new_code_details= filing_team_PINN(user_question,
                                                                                                filename,
                                                                                                initial_code,
                                                                                                naming_agent,
                                                                                                requires_name,
                                                                                                new_code_details,
                                                                                                base_results_dir=base_results_dir)
                        requires_name=False 
                        success,log_file_name=run_code(project_dir,filename,final_filepath)      
                        status, debugger_report,debug_approval,full_log_for_agent,log_content=debugging_team_PINN(user_question,
                                                                                                        final_filepath,
                                                                                                        log_file_name,
                                                                                                        debugger_agent)       

                        if debug_approval:
                                print("--- 5. Calling Advisor Team ---")

                                specific_requests='' 
                                
                                error_report, is_approved, run_summary = advisor_team(
                                                                                user_question=user_question,
                                                                                history_log_string=history_log_string, 
                                                                                project_dir=project_dir,
                                                                                log_content=log_content,
                                                                                strategist_report=strategist_report,
                                                                                implementation_details=implementation_details,
                                                                                advisor_agent=advisor_agent,
                                                                                advisor_parser=advisor_parser,
                                                                                register_agent_parser=register_agent_parser,
                                                                                max_width=800,
                                                                                )
                                run_summary['implementation_details'] = implementation_details
                                run_summary['project_dir'] = project_dir    
                                run_summary['refinement_iterations'] = refinement_iterations       
                                run_summary['data_saving'] = generated_data_details                           
                                history.append(run_summary) 
                                history_log_string = update_and_save_report(
                                        history=history,
                                        user_question=user_question,
                                        strategist_report=strategist_report,
                                        error_report=error_report, # This is the advisor's report
                                        project_dir=project_dir
                                )
                                if run_summary['change_architecture']:
                                        print("--- Looking for a new code template---")
                                        initial_code=''

                
                        else:
                                error_report=f"Debugger Analysis:\n{debugger_report}\n"        
                else:
                        debug_approval=False
                        discrepancies=response_inspector['discrepancies']
                        debugger_report = 'There are discrepancies with the base strategy:\n' + discrepancies     
                        print('Report:',debugger_report)               
                        error_report=f"Debugger Analysis:\n{debugger_report}\n"       
                
        else:
                print('The agents did not get into an agreement')
                break
        print("--- Clearing agent short-term memories ---")
        strategist_agent.clear_history()
        critic_agent.clear_history()
        retriever_agent.clear_history()
        planner_agent.clear_history()
        validator_agent.clear_history()
        advisor_agent.clear_history()
        refinement_iterations=refinement_iterations+1



--- Code Creation Iteration: 0 ---

--- CURRENT TRIAL HISTORY ---
No trials have been run yet.
---------------------------

--- 1. Calling Strategist Team ---

=== 🧠 Starting Strategy Team (Proposer-Critic Loop) ===

--- 🔄 Strategy Iteration 1/5 ---
--- Strategy Report Received ---
### Analysis Phase
1.  **Physics:** The problem describes the **Steady-State Incompressible Navier-Stokes equations**. This is a system of non-linear elliptic partial differential equations. The domain is a standard lid-driven cavity but with a **regularized** top lid velocity (using hyperbolic cosine functions) to avoid corner singularities.
2.  **Feasibility Check:**
    *   **Method of Characteristics:** Not applicable for steady-state elliptic boundary value problems.
    *   **Time-Dependent Solvers:** While one could march a time-dependent solver to steady state, it is inefficient for a direct steady-state request.
    *   **Steady Continuous Galerkin (FEM):** This is the robust, standard approach for